# MWE: gradient CLEAN on AMI data

This notebook demonstrates `virgil.imaging.clean`, which builds an image from point components added one at a time, as Högbom's CLEAN does. Classic CLEAN needs calibrated Fourier phases. virgil's version needs only the gradient of the likelihood, so it works directly on the data virgil fits: here, JWST AMI DISCO observables of a star with a faint companion and a fainter knot. You should come away knowing how to run `clean`, how to read its result, and how to refine the fluxes it finds.

First we simulate the data. The scene is a star with a 2% companion and a 1% knot, both placed on pixel centres of the grid we will reconstruct on, and observed with the AMI uv coverage at 4.8 µm with realistic noise. The printout gives the number of data, the size of the beam, and how well the truth itself fits the noisy data, which is the χ² per point that CLEAN should aim for.

In [ ]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
sys.path.insert(0, str(root / "src"))

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpyro.distributions as dist

from virgil.coverage import ami_grid_record
from virgil.fitting import fit
from virgil.imaging import beam, clean, convolve_beam, image_priors
from virgil.likelihood import whitened_residuals
from virgil.models import PointSource, System, circular_support
from virgil.oidata import OIData
from virgil.plotting import plot_data_model_correlation, plot_model, plot_residual_map

npix, scale = 40, 20.0
fov = npix * scale
truth = System(
    star=PointSource(),
    companion=PointSource(dra=-90.0, ddec=70.0, flux=0.02),
    knot=PointSource(dra=130.0, ddec=-110.0, flux=0.01),
)
template = OIData(ami_grid_record(wavelength_m=4.8e-6, rotation_deg=-6.9))
data = template.with_model(truth, key=jax.random.PRNGKey(4))
resolution = beam(data)
print(
    f"{data.n_independent} DISCO observables; beam {resolution.major_mas:.0f} × "
    f"{resolution.minor_mas:.0f} mas; truth χ²/N = "
    f"{jnp.mean(whitened_residuals(truth, data) ** 2):.3f}"
)

Now we run CLEAN. The components are added to a fixed base scene, the star at flux 1, so their fluxes are relative to the star's. The support allows components anywhere in the field except within half a beam of the star, where their light would be almost indistinguishable from the star's own. CLEAN stops when χ² per data point reaches one, the discrepancy principle. The printout lists why it stopped, how much flux it found, and where its brightest components are.

In [ ]:
support = circular_support(
    npix, scale, radius_mas=fov / 2, inner_radius_mas=0.5 * resolution.minor_mas
)
result = clean(data, npix, scale, base=PointSource(), support=support)
components = result.components
offsets = (jnp.arange(npix) - (npix - 1) / 2) * scale
lines = [
    f"stopped ({result.stop}) after {len(result.chi2_red) - 1} iterations at "
    f"χ²/N = {float(result.chi2_red[-1]):.3f}; total flux "
    f"{float(components.sum()):.4f} (truth 0.030)"
]
for index in jnp.argsort(components.ravel())[::-1][:2]:
    row, col = jnp.unravel_index(index, components.shape)
    lines.append(
        f"component at dra = {float(-offsets[col]):+.0f} mas, ddec = "
        f"{float(-offsets[row]):+.0f} mas: flux {float(components[row, col]):.4f}"
    )
print("\n".join(lines))

Here is the reconstruction beside the truth. Both are drawn without the star, which would swamp them, and convolved with the beam (the shaded ellipse), since detail finer than the beam is not constrained by the data. This beam-convolved image is the "restored" image of radio astronomy. The right panel is the signed difference between the two, in units of the star's flux per pixel, with each image scaled to its total flux. The image is a point estimate with no uncertainties, so the residuals are plain differences, not z-scores. They peak at about 6% of the brightest pixel of the truth, at the sources themselves, where CLEAN's flux is distributed a little differently among neighbouring pixels.

In [ ]:
faint = System(companion=truth.companion, knot=truth.knot)
# render() gives unit-flux images: scale each by its flux relative to the star.
truth_smooth = 0.03 * convolve_beam(faint.render(npix, fov), scale, resolution)
clean_smooth = float(components.sum()) * convolve_beam(
    result.model.clean.render(npix, fov), scale, resolution
)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
plot_model(faint, fov, npix, ax=axes[0], beam=resolution, convolve=True, title="truth ⊛ beam")
plot_model(result.model.clean, fov, npix, ax=axes[1], beam=resolution, convolve=True, title="CLEAN, restored")
plot_residual_map(clean_smooth - truth_smooth, fov, ax=axes[2], title="CLEAN − truth")
plt.tight_layout()
plt.show()

An image that looks right should also fit the data. This correlation plot compares the CLEAN model's predictions with the data, with the data's error bars and a one-to-one line. AMI DISCO observables are projections that mix amplitude and phase information, so they appear in a single panel. Points scattered evenly about the line across the full range show that the components explain the data.

In [ ]:
n_vis = data.vis.size
prediction = data.model(result.model)
summary = {
    "CLEAN": {
        "vis_mean": prediction[:n_vis],
        "vis_std": 0 * prediction[:n_vis],
        "phi_mean": prediction[n_vis:],
        "phi_std": 0 * prediction[n_vis:],
    }
}
fig, (ax, _) = plot_data_model_correlation(data, summary, figsize=(5, 4.5))
ax.set_title("CLEAN model against the simulated DISCOs")
plt.show()

CLEAN adds only a fraction of the flux the data ask for at each step and stops as soon as χ² per point reaches one, so its total flux is usually a little low, and a source can be shared among neighbouring pixels. Because `result.model` is an ordinary `System` containing an `Image` whose support is the components, `fit` can refit the component fluxes directly. That is a small problem, a dozen or so pixels rather than the whole grid, so it needs no regulariser.

In [ ]:
priors = image_priors(result.model) | {"clean.flux": dist.Uniform(0.0, 1.0)}
polished = fit(result.model, priors, data)
print(
    f"total flux: {float(components.sum()):.4f} after CLEAN, "
    f"{float(polished.model.clean.flux):.4f} after refitting (truth 0.0300); "
    f"χ²/N = {polished.info['chi2_red']:.3f}"
)

In summary, `clean(data, npix, pixel_scale_mas, base=...)` found the companion and the knot directly from AMI DISCO observables, without the self-calibration that classic CLEAN would need. Its result is an ordinary virgil model, so it can be shown restored with the beam, checked against the data like any other fit, and refined with `fit`. The tutorial "Imaging, part 6: sparse images and CLEAN" covers the loop gain and closure-phase data as well.